## INDEX

- [1. Introduction](#introduction)
- [2. Objective](#objective)
- [3. Data Loading](#data-loading)
- [4. Data Preparation](#data-preparation)
- [5. Handling Missing & Special Values](#missing-values)
- [6. Feature Transformation](#feature-transformation)
- [7. Feature Creation](#feature-creation)
- [8. Encoding](#encoding)
- [9. Feature Selection](#feature-selection)
- [10. Final Dataset](#final-dataset)

<a id="introduction"></a>
## 1. Introduction

This notebook focuses on transforming raw data into a format suitable for machine learning models. It builds upon the insights obtained during the Exploratory Data Analysis (EDA) phase.

<a id="objective"></a>
## 2. Objective

**Objective:** Prepare the dataset for modeling by handling missing values, transforming variables, creating new features, and encoding categorical variables.


<a id="data-loading"></a>
## 3. Data Loading

In [3]:
import pandas as pd
import numpy as np

csv = '../data/bank-additional-full.csv'
df = pd.read_csv(csv, sep=";")
df.head()

,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
2,37,services,married,high.school,no,yes,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
4,56,services,married,high.school,no,no,yes,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no


<a id="data-preparation"></a>
## 4. Data Preparation

In [4]:
df_fe = df.copy()

A working copy of the dataset is created to ensure that all transformations applied during the feature engineering process do not modify the original data.

This approach allows for reproducibility and prevents unintended side effects during experimentation.

<a id="missing-values"></a>
## 5. Handling Missing & Special Values

This step focuses on identifying and handling special values that may affect model performance.

During EDA, it was observed that certain variables contain encoded missing values rather than true nulls. These values must be properly treated to ensure meaningful feature representation.

In particular:
- The `pdays` variable uses the value 999 to indicate that the client was not previously contacted.
- Several categorical variables contain the value "unknown", which represents missing or undisclosed information.

These cases are handled explicitly to improve data quality and model interpretability.

In [5]:
df_fe['pdays'] = df_fe['pdays'].replace(999, np.nan)

The value 999 in `pdays` is treated as a missing value, as it represents clients who were not previously contacted rather than a true numerical value.

In [6]:
cat_cols = ['job', 'marital', 'education', 'default', 'housing', 'loan']

for col in cat_cols:
    df_fe[col] = df_fe[col].replace('unknown', 'missing')

Categorical variables containing the value "unknown" are recoded as "missing" to explicitly represent the absence of information.

This allows the model to treat missingness as a potential signal rather than ignoring it.

<a id="feature-transformation"></a>
## 6. Feature Transformation

In this step, variables are transformed to improve their suitability for modeling.

Transformations are applied to address skewness, scale differences, and to convert variables into formats that better capture their underlying patterns.

Particular attention is given to numerical variables with highly skewed distributions, as well as to the target variable.

## 6.1 Target Binario

In [7]:
df_fe['y_bin'] = df_fe['y'].map({'no': 0, 'yes': 1})

The target variable is transformed into a binary format to facilitate modeling, where 1 represents a successful subscription and 0 represents no subscription.

## 6.2 Transformaciones logaritmicas

In [8]:
df_fe['duration_log'] = np.log1p(df_fe['duration'])
df_fe['campaign_log'] = np.log1p(df_fe['campaign'])

Logarithmic transformations are applied to skewed numerical variables such as `duration` and `campaign` to reduce the impact of extreme values and improve distribution symmetry.

This helps stabilize model behavior and can improve performance for algorithms sensitive to scale and distribution.

<a id="feature-creation"></a>
## 7. Feature Creation

In this step, new features are created based on patterns identified during the EDA phase.

These features aim to capture meaningful behavioral signals, simplify complex relationships, and improve model interpretability.

The focus is on transforming raw variables into more informative representations that better reflect client interaction history and campaign dynamics.

In [9]:
df_fe['was_contacted_before'] = (df_fe['previous'] > 0).astype(int)

The `was_contacted_before` feature indicates whether the client has been contacted in previous campaigns. This captures prior engagement and helps distinguish new clients from previously targeted ones.

In [10]:
df_fe['multiple_contacts'] = (df_fe['campaign'] > 1).astype(int)

The `multiple_contacts` feature identifies whether a client was contacted more than once during the campaign, capturing potential saturation or diminishing returns from repeated contact attempts.

In [11]:
df_fe['recent_contact'] = (df_fe['pdays'] < 30).astype(int)

The `recent_contact` feature captures whether the client was contacted recently, which may indicate higher engagement or recall, potentially increasing the likelihood of conversion.

In [12]:
df_fe['has_history'] = (
    (df_fe['previous'] > 0) | (df_fe['pdays'].notna())
).astype(int)

The `has_history` feature aggregates multiple signals related to prior interactions, providing a simplified representation of whether the client has any form of historical engagement.

These engineered features are based on behavioral insights identified during EDA and are expected to enhance model performance by capturing patterns not directly observable in the raw variables.

<a id="encoding"></a>
## 8. Encoding Categorical Variables

Categorical variables must be converted into numerical format to be used in machine learning models.

In this step, one-hot encoding is applied to transform categorical features into binary indicator variables. This approach allows the model to interpret each category independently without introducing ordinal relationships.

To avoid multicollinearity, one category per variable is dropped.

In [13]:
df_model = pd.get_dummies(df_fe, drop_first=True)

One-hot encoding is applied to all categorical variables, converting each category into a separate binary feature. 

The `drop_first=True` parameter is used to prevent redundancy and reduce the risk of multicollinearity.

In [14]:
df_model.shape
df_model.head()

,age,duration,campaign,pdays,previous,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,...,month_nov,month_oct,month_sep,day_of_week_mon,day_of_week_thu,day_of_week_tue,day_of_week_wed,poutcome_nonexistent,poutcome_success,y_yes
0,56,261,1,NaN,0,1.1,93.994,-36.4,4.857,5191.0,...,0,0,0,1,0,0,0,1,0,0
1,57,149,1,NaN,0,1.1,93.994,-36.4,4.857,5191.0,...,0,0,0,1,0,0,0,1,0,0
2,37,226,1,NaN,0,1.1,93.994,-36.4,4.857,5191.0,...,0,0,0,1,0,0,0,1,0,0
3,40,151,1,NaN,0,1.1,93.994,-36.4,4.857,5191.0,...,0,0,0,1,0,0,0,1,0,0
4,56,307,1,NaN,0,1.1,93.994,-36.4,4.857,5191.0,...,0,0,0,1,0,0,0,1,0,0


<a id="feature-selection"></a>
## 9. Feature Selection


<a id="final-dataset"></a>
## 10. Final Dataset for Modeling